# Day 2: Preprocessing & Feature Engineering
Fix TotalCharges dtype, encode categoricals, scale numerics, engineer new features.

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)
df.shape

(7043, 21)

In [23]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isnull().sum()

np.int64(11)

In [24]:
df[df["TotalCharges"].isnull()][["tenure", "MonthlyCharges", "TotalCharges"]]


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


In [25]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["TotalCharges"].isnull().sum()

np.int64(0)

In [26]:
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes":1, "No":0})
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [27]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(col, "->", df[col].unique())

gender -> <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Partner -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines -> <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService -> <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
OnlineSecurity -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup -> <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingTV -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingMovies -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Contract -> <StringArray>
['Mont

C:\Users\aditi\AppData\Local\Temp\ipykernel_34560\3738988987.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


In [28]:
col_to_fix = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for col in col_to_fix:
    df[col] = df[col].replace({"No internet service" : "No", "No phone service" : "No"})

for col in col_to_fix:
    print(col, "->", df[col].unique())

MultipleLines -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineSecurity -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineBackup -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
DeviceProtection -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
TechSupport -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingMovies -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [29]:
binary_cols = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"] + col_to_fix
for col in binary_cols:
    print(col, df[col].unique())

gender <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Partner <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PaperlessBilling <StringArray>
['Yes', 'No']
Length: 2, dtype: str
MultipleLines <StringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineSecurity <StringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineBackup <StringArray>
['Yes', 'No']
Length: 2, dtype: str
DeviceProtection <StringArray>
['No', 'Yes']
Length: 2, dtype: str
TechSupport <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingMovies <StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [30]:
binary_map_yesno = {"Yes" : 1, "No" : 0}
gender_map = {"Female" : 0, "Male" : 1}

for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"] + col_to_fix:
    df[col] = df[col].map(binary_map_yesno)

df["gender"] = df["gender"].map(gender_map)
df[binary_cols].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,0,1,0,0,1,0,0,1,0,0,0,0
1,1,0,0,1,0,0,1,0,1,0,0,0
2,1,0,0,1,1,0,1,1,0,0,0,0
3,1,0,0,0,0,0,1,0,1,1,0,0
4,0,0,0,1,1,0,0,0,0,0,0,0


In [31]:
#one hot encoding for multi-category columns

df = pd.get_dummies(df, columns=["InternetService", "Contract", "PaymentMethod"], drop_first = True)
df.shape

(7043, 24)

In [32]:
df.dtypes.value_counts()

int64      15
bool        7
float64     2
Name: count, dtype: int64

In [34]:
#Total number of subscribed add-on services
service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["TotalServices"] = df[service_cols].sum(axis = 1)


#Tenure buckets- captures non -linear churn risk by customer lifecycle stagesabs
df["TenureGroup"] = pd.cut(df["tenure"], bins=[0,12,24,48,72], labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr"])
df[["tenure", "TenureGroup", "TotalServices"]].head()

,tenure,TenureGroup,TotalServices
0,1,0-1yr,1
1,34,2-4yr,2
2,2,0-1yr,2
3,45,2-4yr,3
4,2,0-1yr,0


In [35]:
#encoding TentureGroup
df = pd.get_dummies(df, columns=["TenureGroup"], drop_first=True)
df.shape

(7043, 28)

In [36]:
OUTPUT_PATH = Path("../data/processed/telco_churn_processed.csv")
df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH, "| Shape:", df.shape)

Saved: ..\data\processed\telco_churn_processed.csv | Shape: (7043, 28)
